# 04 — Phân tích khám phá & trực quan hóa khách hàng rời bỏ Cell2Cell

**Vai trò:** chuyên viên phân tích dữ liệu của công ty viễn thông Cell2Cell.

**Bài toán kinh doanh:** trong mẫu dữ liệu, gần 3 trong 10 khách rời bỏ dịch vụ. Ban giám đốc cần biết:

1. Tình trạng rời bỏ phổ biến đến đâu và gây thiệt hại bao nhiêu?
2. Tín hiệu nào (vòng đời, thiết bị, hành vi dịch vụ, địa lý, nhân khẩu) đi cùng rời bỏ?
3. Ai đang có nguy cơ rời bỏ và nên ưu tiên giữ ai?
4. Nên thử chính sách giữ chân nào?

**Cách chạy:** chạy `python work/viz/prepare_viz_data.py` trước (tạo `work/build/viz_customers.parquet`), sau đó Khởi động lại nhân tính toán → Chạy tất cả các ô.
Mọi biểu đồ được vẽ bằng hàm trong `work/viz/charts.py`, cùng thư viện với bảng điều khiển, nên số liệu trên sổ phân tích và bảng điều khiển luôn khớp.
Tiêu đề mỗi biểu đồ là một **câu kết luận** sinh tự động từ số liệu.

**B?n EDA ??y ??:** kh?i ph?c n?i dung t? commit `4026cfe`, g?m c?c bi?u ?? d?ch v?, khu v?c v? nh?n kh?u h?c ban ??u. M? h?nh v?n d?ng 10 bi?n; EDA c? th?m c?c tr??ng kh?o s?t. Code bi?u ?? v? x? l? d? li?u ?i k?m trong file n?y, ??c l?p v?i b?n `notebook4_10bien.ipynb`. K?t qu? ch? hi?n th?/l?u trong notebook.

## 1. Thiết lập

In [ ]:
from pathlib import Path
from base64 import b64encode
import json

from IPython.display import Markdown, display

import pandas as pd

# Tìm thư mục gốc dự án từ thư mục làm việc (chạy được từ root hoặc từ work/viz).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "cell2celltrain.csv").exists())

pd.set_option("display.max_columns", 20)
dashboard_figures = {}

def show(key):
    """Vẽ biểu đồ theo mã trong charts.FIGURES và lưu PNG vào reports/figures."""
    fig = c.FIGURES[key](df)
    dashboard_figures[key] = fig
    image_bytes = fig.to_image(format="png", width=1000, scale=1.5)
    display({
        "application/vnd.plotly.v1+json": json.loads(fig.to_json()),
        "image/png": b64encode(image_bytes).decode("ascii"),
    }, raw=True)


def explain(key):
    title = dashboard_figures[key].layout.title.text.split("<br>")[0].replace("<b>", "").replace("</b>", "")
    display(Markdown("**Nhận xét:** " + title + "."))


In [ ]:
# Th? vi?n bi?u ?? nguy?n b?n t? commit 4026cfe.
# Namespace ri?ng gi?p gi? nguy?n h?m c? v? kh?ng ?nh h??ng notebook 10 bi?n.
from types import ModuleType

chart_source = r'''"""Hàm vẽ dùng chung cho notebook 04 và dashboard. Mỗi hàm nhận DataFrame đã lọc, trả về go.Figure.

Tiêu đề biểu đồ là action title sinh từ số liệu truyền vào, không hard-code con số.
"""

from __future__ import annotations

from textwrap import wrap

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CHURN = "#D64545"
STAY = "#8FA3B5"
ACCENT = "#1F6FB2"
NEUTRAL = "#C9D3DC"
INK = "#1D2B36"
MUTED = "#5B6B78"
GRID = "#EDF1F4"
FONT = "Inter, Segoe UI, Arial, sans-serif"

TARGET = "ChurnFlag"
MIN_N = 100  # nhóm nhỏ hơn ngưỡng này không đủ tin cậy để so sánh rate

CREDIT_ORDER = ["1-Highest", "2-High", "3-Good", "4-Medium", "5-Low", "6-VeryLow", "7-Lowest"]

FIELD_LABELS = {
    "CustomerID": "Mã khách hàng", "Churn": "Trạng thái rời bỏ", "ChurnFlag": "Rời bỏ",
    "MonthlyRevenue": "Doanh thu tháng", "MonthlyMinutes": "Phút gọi mỗi tháng",
    "TotalRecurringCharge": "Phí định kỳ", "DirectorAssistedCalls": "Cuộc gọi qua tổng đài hỗ trợ",
    "OverageMinutes": "Phút vượt gói", "RoamingCalls": "Cuộc gọi chuyển vùng",
    "PercChangeMinutes": "Thay đổi phút gọi", "PercChangeRevenues": "Thay đổi doanh thu",
    "DroppedCalls": "Cuộc gọi bị rớt", "BlockedCalls": "Cuộc gọi bị chặn",
    "UnansweredCalls": "Cuộc gọi không trả lời", "CustomerCareCalls": "Cuộc gọi chăm sóc khách hàng",
    "ThreewayCalls": "Cuộc gọi ba bên", "ReceivedCalls": "Cuộc gọi nhận được",
    "OutboundCalls": "Cuộc gọi đi", "InboundCalls": "Cuộc gọi đến",
    "PeakCallsInOut": "Cuộc gọi giờ cao điểm", "OffPeakCallsInOut": "Cuộc gọi ngoài giờ cao điểm",
    "DroppedBlockedCalls": "Cuộc gọi rớt hoặc bị chặn", "CallForwardingCalls": "Cuộc gọi chuyển tiếp",
    "CallWaitingCalls": "Cuộc gọi chờ", "MonthsInService": "Tháng sử dụng dịch vụ",
    "UniqueSubs": "Số thuê bao đã đăng ký", "ActiveSubs": "Số thuê bao hoạt động",
    "ServiceArea": "Mã vùng phục vụ", "Handsets": "Số thiết bị", "HandsetModels": "Số mẫu thiết bị",
    "CurrentEquipmentDays": "Tuổi thiết bị", "AgeHH1": "Tuổi thành viên thứ nhất",
    "AgeHH2": "Tuổi thành viên thứ hai", "ChildrenInHH": "Có trẻ em trong hộ",
    "HandsetRefurbished": "Thiết bị tân trang", "HandsetWebCapable": "Thiết bị hỗ trợ mạng",
    "TruckOwner": "Sở hữu xe tải", "RVOwner": "Sở hữu xe du lịch lưu động",
    "Homeownership": "Tình trạng sở hữu nhà", "BuysViaMailOrder": "Mua hàng qua thư",
    "RespondsToMailOffers": "Phản hồi ưu đãi qua thư", "OptOutMailings": "Từ chối nhận thư",
    "NonUSTravel": "Du lịch ngoài Hoa Kỳ", "OwnsComputer": "Sở hữu máy tính",
    "HasCreditCard": "Có thẻ tín dụng", "RetentionCalls": "Cuộc gọi giữ chân",
    "RetentionOffersAccepted": "Số ưu đãi giữ chân được nhận", "NewCellphoneUser": "Người mới dùng điện thoại",
    "NotNewCellphoneUser": "Người đã dùng điện thoại", "ReferralsMadeBySubscriber": "Số lượt giới thiệu",
    "IncomeGroup": "Nhóm thu nhập", "OwnsMotorcycle": "Sở hữu xe máy",
    "AdjustmentsToCreditRating": "Điều chỉnh hạng tín dụng", "HandsetPrice": "Giá thiết bị",
    "MadeCallToRetentionTeam": "Đã gọi bộ phận giữ chân", "CreditRating": "Hạng tín dụng",
    "PrizmCode": "Khu vực sinh sống", "Occupation": "Nghề nghiệp", "MaritalStatus": "Đã kết hôn",
    "Region": "Vùng", "RetentionOfferAccepted": "Đã nhận ưu đãi giữ chân",
    "TenureBand": "Nhóm thời gian sử dụng", "EquipmentAgeBand": "Nhóm tuổi thiết bị",
    "CareCallsBand": "Nhóm cuộc gọi chăm sóc", "DroppedBand": "Nhóm cuộc gọi rớt",
    "OverageBand": "Nhóm phút vượt gói", "MinutesBand": "Nhóm phút gọi",
    "ActionSegment": "Phân khúc hành động", "RiskScore": "Điểm rủi ro", "RiskDecile": "Nhóm rủi ro",
    "n": "Số khách", "churners": "Khách rời bỏ", "rate": "Tỷ lệ rời bỏ",
    "revenue": "Tổng doanh thu tháng", "avg_revenue": "Doanh thu tháng trung bình",
    "revenue_at_risk": "Doanh thu tháng bị đe dọa",
}
VALUE_LABELS = {
    "CreditRating": dict(zip(CREDIT_ORDER, ["1-Cao nhất", "2-Cao", "3-Tốt", "4-Trung bình",
                                          "5-Thấp", "6-Rất thấp", "7-Thấp nhất"])),
    "Occupation": {"Clerical": "Văn phòng", "Crafts": "Thủ công", "Homemaker": "Nội trợ",
                   "Other": "Khác", "Professional": "Chuyên môn", "Retired": "Nghỉ hưu",
                   "Self": "Tự kinh doanh", "Student": "Sinh viên"},
    "PrizmCode": {"Other": "Khác", "Rural": "Nông thôn", "Suburban": "Ngoại ô", "Town": "Thị trấn"},
    "Churn": {"Yes": "Rời bỏ", "No": "Ở lại"},
    "Homeownership": {"Known": "Đã xác định"},
}
SUMMARY_LABELS = {"count": "Số giá trị", "mean": "Trung bình", "std": "Độ lệch chuẩn",
                  "min": "Nhỏ nhất", "max": "Lớn nhất", "50%": "Trung vị",
                  "unique": "Số giá trị khác nhau", "top": "Phổ biến nhất", "freq": "Tần suất"}


def field_label(column) -> str:
    return FIELD_LABELS.get(column, str(column))


def display_value(column, value) -> str:
    if pd.isna(value):
        return "Không rõ"
    value = str(value)
    return VALUE_LABELS.get(column, {}).get(value, {"Yes": "Có", "No": "Không", "Unknown": "Không rõ"}.get(value, value))


def display_table(frame: pd.DataFrame) -> pd.DataFrame:
    """Việt hóa bảng trình bày trên bản sao, giữ nguyên bảng dùng để tính toán."""
    result = frame.copy()
    for column in result:
        if pd.api.types.is_object_dtype(result[column]) or isinstance(result[column].dtype, pd.CategoricalDtype):
            result[column] = result[column].astype(object).map(lambda value: display_value(column, value))
    labels = {**FIELD_LABELS, **SUMMARY_LABELS}
    return result.rename(columns=labels, index=labels)


# ---------- định dạng ----------

def fmt_n(value) -> str:
    return f"{int(round(value)):,}".replace(",", ".")


def fmt_pct(value, digits=1) -> str:
    return f"{100 * value:.{digits}f}%".replace(".", ",")


def fmt_money(value) -> str:
    if abs(value) >= 1e6:
        return f"${value / 1e6:.2f} triệu".replace(".", ",")
    if abs(value) >= 1e3:
        return f"${value / 1e3:.1f} nghìn".replace(".", ",")
    return f"${value:.1f}".replace(".", ",")


def fmt_ratio(value) -> str:
    return f"{value:.1f}".replace(".", ",")


def baseline(df: pd.DataFrame) -> float:
    return float(df[TARGET].mean()) if len(df) else float("nan")


# ---------- tổng hợp ----------

def rate_by(df: pd.DataFrame, col, order=None, min_n=0) -> pd.DataFrame:
    """Bảng [col, n, churners, rate]; rate = churner / n trong cùng nhóm."""
    table = (df.groupby(col, observed=True, dropna=False)[TARGET]
             .agg(n="size", churners="sum").reset_index())
    table["rate"] = table.churners / table.n
    table = table[table.n >= min_n]
    if order is not None:
        table[col] = pd.Categorical(table[col], categories=[o for o in order if o in set(table[col])], ordered=True)
        table = table.sort_values(col)
    return table.reset_index(drop=True)


def kpis(df: pd.DataFrame) -> dict:
    churned = df[df[TARGET] == 1]
    out = {
        "customers": len(df),
        "churners": int(df[TARGET].sum()),
        "rate": baseline(df),
        "lost_revenue": float(churned.MonthlyRevenue.sum()),
        "tenure_mean": float(df.MonthsInService.mean()) if len(df) else float("nan"),
    }
    if "RiskDecile" in df:
        high = df.RiskDecile >= 8
        out["high_risk_share"] = float(high.mean()) if len(df) else float("nan")
        out["high_risk_n"] = int(high.sum())
    return out


def segment_summary(df: pd.DataFrame) -> pd.DataFrame:
    table = (df.groupby("ActionSegment", observed=True)
             .agg(n=(TARGET, "size"), churners=(TARGET, "sum"),
                  revenue=("MonthlyRevenue", "sum"), avg_revenue=("MonthlyRevenue", "mean"))
             .reset_index())
    table["rate"] = table.churners / table.n
    table["revenue_at_risk"] = table.revenue * table.rate
    return table


# ---------- style ----------

def apply_style(fig: go.Figure, title: str, subtitle: str = "", height=380) -> go.Figure:
    text = f"<b>{title}</b>" + (f"<br><span style='font-size:12px;color:{MUTED}'>{subtitle}</span>" if subtitle else "")
    fig.update_layout(
        title=dict(text=text, x=0.01, xanchor="left", y=0.98, yanchor="top", pad=dict(t=8), font=dict(size=15, color=INK)),
        template="plotly_white", height=height, separators=",.",
        font=dict(family=FONT, size=12, color=INK),
        margin=dict(l=10, r=20, t=92, b=40),
        paper_bgcolor="white", plot_bgcolor="white",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1, xanchor="right", title=None),
        hoverlabel=dict(bgcolor="white", font=dict(family=FONT, color=INK)),
    )
    fig.update_xaxes(gridcolor=GRID, zeroline=False, automargin=True, title_font=dict(color=MUTED))
    fig.update_yaxes(gridcolor=GRID, zeroline=False, automargin=True, title_font=dict(color=MUTED))
    return fig


def add_baseline(fig: go.Figure, base: float, axis="y", **kwargs) -> go.Figure:
    line = dict(line_dash="dash", line_color=MUTED, line_width=1,
                annotation_text=f"Trung bình {fmt_pct(base)}", annotation_font_color=MUTED,
                annotation_font_size=11, **kwargs)
    if axis == "y":
        fig.add_hline(y=base, annotation_position="top left", **line)
    else:
        fig.add_vline(x=base, annotation_position="top", **line)
    return fig


def _hover(table: pd.DataFrame) -> np.ndarray:
    return np.stack([table.n.map(fmt_n), table.churners.map(fmt_n), table.rate.map(fmt_pct)], axis=-1)


HOVER = "<b>%{x}</b><br>Số khách: %{customdata[0]}<br>Khách rời bỏ: %{customdata[1]}<br>Tỷ lệ rời bỏ: %{customdata[2]}<extra></extra>"
HOVER_H = HOVER.replace("%{x}", "%{y}")


def empty_figure(message="Không có dữ liệu cho bộ lọc này") -> go.Figure:
    fig = go.Figure()
    fig.add_annotation(text=message, showarrow=False, font=dict(color=MUTED, size=14))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return apply_style(fig, "")


# ---------- C01–C02: tổng quan ----------

def churn_donut(df: pd.DataFrame) -> go.Figure:
    k = kpis(df)
    fig = go.Figure(go.Pie(
        labels=["Rời bỏ", "Ở lại"], values=[k["churners"], k["customers"] - k["churners"]],
        hole=0.62, sort=False, marker=dict(colors=[CHURN, STAY]), textinfo="percent",
        hovertemplate="%{label}: %{value:,.0f} khách<extra></extra>"))
    fig.add_annotation(text=f"<b>{fmt_pct(k['rate'])}</b><br>rời bỏ", showarrow=False, font=dict(size=20, color=INK))
    one_in = 1 / k["rate"] if k["rate"] else float("nan")
    return apply_style(fig, f"Cứ khoảng {fmt_ratio(one_in)} khách thì có 1 khách rời bỏ",
                       f"{fmt_n(k['churners'])} / {fmt_n(k['customers'])} khách", height=460)


def missing_bar(df: pd.DataFrame, columns) -> go.Figure:
    share = df[columns].isna().mean().loc[lambda s: s.gt(0)].sort_values()
    if share.empty:
        return empty_figure("Không có giá trị thiếu trong lát cắt này")
    fig = go.Figure(go.Bar(x=share.values, y=[field_label(c) for c in share.index], orientation="h", marker_color=ACCENT,
                           text=[fmt_pct(v, 2) for v in share.values], textposition="outside",
                           hovertemplate="%{y}: %{x:.2%}<extra></extra>"))
    fig.update_xaxes(tickformat=".1%", title="% dòng bị thiếu", range=[0, share.max() * 1.25])
    return apply_style(fig, f"Dữ liệu thiếu rất ít: cao nhất là {field_label(share.index[-1]).lower()} ({fmt_pct(share.iloc[-1], 2)})",
                       f"{len(share)} / {len(columns)} cột có giá trị thiếu", height=max(300, 28 * len(share) + 120))


# ---------- C03–C04: phân bố ----------

def distribution_overlay(df: pd.DataFrame, col: str, label: str, clip=0.99) -> go.Figure:
    upper = df[col].quantile(clip)
    data = df[df[col].le(upper)]
    fig = go.Figure()
    for flag, name, color in [(0, "Ở lại", STAY), (1, "Rời bỏ", CHURN)]:
        fig.add_trace(go.Histogram(x=data.loc[data[TARGET] == flag, col], name=name, marker_color=color,
                                   opacity=0.6, histnorm="percent", nbinsx=40))
    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title=label)
    fig.update_yaxes(title="% khách trong nhóm")
    med = df.groupby(TARGET)[col].median()
    direction = "thấp hơn" if med.get(1, 0) < med.get(0, 0) else "cao hơn"
    return apply_style(fig, f"Khách rời bỏ có {label.lower()} trung vị {direction} khách ở lại",
                       f"Trung vị: rời bỏ {fmt_ratio(med.get(1, np.nan))} · ở lại {fmt_ratio(med.get(0, np.nan))} "
                       f"· bỏ 1% giá trị lớn nhất cho dễ nhìn")


def box_by_churn(df: pd.DataFrame, cols: dict) -> go.Figure:
    fig = make_subplots(rows=1, cols=len(cols), subplot_titles=list(cols.values()))
    for i, col in enumerate(cols, start=1):
        for flag, name, color in [(0, "Ở lại", STAY), (1, "Rời bỏ", CHURN)]:
            fig.add_trace(go.Box(y=df.loc[df[TARGET] == flag, col], name=name, marker_color=color,
                                 boxpoints=False, showlegend=i == 1, legendgroup=name), row=1, col=i)
    med = df.groupby(TARGET).CurrentEquipmentDays.median() if "CurrentEquipmentDays" in cols else None
    title = ("Khách rời bỏ dùng máy cũ hơn rõ rệt" if med is not None and med.get(1, 0) > med.get(0, 0)
             else "So sánh phân bố giữa khách rời bỏ và ở lại")
    sub = (f"Trung vị tuổi máy: rời bỏ {fmt_n(med.get(1, 0))} ngày · ở lại {fmt_n(med.get(0, 0))} ngày"
           if med is not None else "")
    fig = apply_style(fig, title, sub, height=420)
    fig.update_annotations(font=dict(size=12, color=MUTED))
    return fig


# ---------- C05–C07: vòng đời và thiết bị ----------

def tenure_line(df: pd.DataFrame, max_month=48) -> go.Figure:
    table = rate_by(df[df.MonthsInService.le(max_month)], "MonthsInService", min_n=MIN_N)
    if table.empty:
        return empty_figure()
    base = baseline(df)
    peak = table.loc[table.rate.idxmax()]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=table.MonthsInService, y=table.rate, mode="lines+markers", name="Tỷ lệ rời bỏ",
                             line=dict(color=CHURN, width=2.5), marker=dict(size=5), customdata=_hover(table),
                             hovertemplate=HOVER.replace("%{x}", "Tháng %{x}")))
    fig.add_hline(y=base, line_dash="dash", line_color=MUTED, line_width=1,
                  annotation_text=f"Trung bình {fmt_pct(base)}", annotation_position="bottom right",
                  annotation_font=dict(color=MUTED, size=11))
    fig.update_yaxes(tickformat=".0%", title="Tỷ lệ rời bỏ", rangemode="tozero")
    fig.update_xaxes(title="Số tháng sử dụng dịch vụ", dtick=6)
    fig = apply_style(fig, f"Tỷ lệ rời bỏ vọt lên {fmt_pct(peak.rate)} ở tháng {int(peak.MonthsInService)}: thời điểm hết hợp đồng 1 năm",
                      f"Tỷ lệ rời bỏ theo từng tháng (chỉ tháng có ≥{MIN_N} khách, tối đa {max_month} tháng)", height=460)
    fig.update_layout(showlegend=False)
    return fig


def tenure_stacked(df: pd.DataFrame) -> go.Figure:
    table = rate_by(df.dropna(subset=["MonthsInService"]), "MonthsInService").sort_values("MonthsInService")
    if table.empty:
        return empty_figure()
    peak = table.loc[table.churners.idxmax()]
    fig = go.Figure()
    for name, counts, color in [
        ("Rời bỏ", table.churners, CHURN),
        ("Ở lại", table.n - table.churners, STAY),
    ]:
        fig.add_trace(go.Bar(
            x=table.MonthsInService, y=counts, name=name, marker_color=color,
            customdata=_hover(table),
            hovertemplate="<b>Tháng %{x}</b><br>%{fullData.name}: %{y:,.0f} khách"
                          "<br>Tổng khách: %{customdata[0]}<br>Tỷ lệ rời bỏ: %{customdata[2]}<extra></extra>",
        ))
    fig.update_layout(barmode="stack", bargap=0.16)
    fig.update_xaxes(title="Thời gian sử dụng dịch vụ (tháng)", dtick=6, rangemode="tozero")
    fig.update_yaxes(title="Số khách hàng", rangemode="tozero", separatethousands=True)
    return apply_style(fig, f"Tháng {int(peak.MonthsInService)} có nhiều khách rời bỏ nhất: {fmt_n(peak.churners)} khách",
                       f"Mỗi cột là một tháng · {fmt_n(table.n.sum())} khách · toàn bộ thời gian sử dụng", height=460)


def rate_bar(df: pd.DataFrame, col: str, label: str, order=None, horizontal=False, title=None, min_n=MIN_N,
             base=None) -> go.Figure:
    """Bar tỷ lệ rời bỏ theo nhóm, có baseline; nhóm trên trung bình tô đỏ."""
    table = rate_by(df, col, order=order, min_n=min_n)
    if table.empty:
        return empty_figure()
    if order is None:
        table = table.sort_values("rate", ascending=horizontal)
    base = baseline(df) if base is None else base
    colors = [CHURN if r > base else STAY for r in table.rate]
    labels = table[col].astype(str).map(lambda value: display_value(col, value))
    text = [fmt_pct(r) for r in table.rate]
    if horizontal:
        bar = go.Bar(x=table.rate, y=labels, orientation="h", marker_color=colors, text=text,
                     textposition="outside", customdata=_hover(table), hovertemplate=HOVER_H)
    else:
        bar = go.Bar(x=labels, y=table.rate, marker_color=colors, text=text, textposition="outside",
                     customdata=_hover(table), hovertemplate=HOVER)
    fig = go.Figure(bar)
    hi, lo = table.loc[table.rate.idxmax()], table.loc[table.rate.idxmin()]
    if title is None:
        title = (f"{label} “{display_value(col, hi[col])}” có tỷ lệ rời bỏ {fmt_pct(hi.rate)}, gấp {fmt_ratio(hi.rate / lo.rate)} lần nhóm “{display_value(col, lo[col])}”"
                 if lo.rate > 0 else f"{label} “{display_value(col, hi[col])}” có tỷ lệ rời bỏ cao nhất ({fmt_pct(hi.rate)})")
    rate_axis = dict(tickformat=".0%", range=[0, table.rate.max() * 1.22], title="Tỷ lệ rời bỏ")
    if horizontal:
        fig.update_xaxes(**rate_axis)
        fig.update_yaxes(title=None)
        add_baseline(fig, base, axis="x")
        height = max(340, 26 * len(table) + 130)
    else:
        fig.update_yaxes(**rate_axis)
        fig.update_xaxes(title=label, type="category")
        add_baseline(fig, base)
        height = 380
    return apply_style(fig, title, f"Đỏ = cao hơn trung bình · n = {fmt_n(table.n.sum())} khách", height=height)


def tenure_equipment_heatmap(df: pd.DataFrame) -> go.Figure:
    data = df[~df.EquipmentAgeBand.isin(["Bất thường", "Không rõ"])]
    table = (data.groupby(["TenureBand", "EquipmentAgeBand"], observed=True)[TARGET]
             .agg(n="size", rate="mean").reset_index())
    if table.empty:
        return empty_figure()
    table.loc[table.n < MIN_N, "rate"] = np.nan
    rate = table.pivot(index="TenureBand", columns="EquipmentAgeBand", values="rate")
    count = table.pivot(index="TenureBand", columns="EquipmentAgeBand", values="n").reindex_like(rate)
    text = [[fmt_pct(v) if pd.notna(v) else "" for v in row] for row in rate.values]
    hover = [[f"n = {fmt_n(c)}" if pd.notna(c) else "" for c in row] for row in count.values]
    fig = go.Figure(go.Heatmap(z=rate.values, x=rate.columns.astype(str), y=rate.index.astype(str),
                               colorscale="Reds", text=text, texttemplate="%{text}", customdata=hover,
                               hovertemplate="%{y} · máy %{x}<br>Tỷ lệ rời bỏ: %{z:.1%}<br>%{customdata}<extra></extra>",
                               colorbar=dict(tickformat=".0%", title="Tỷ lệ rời bỏ")))
    stacked = table.dropna(subset=["rate"])
    if stacked.empty:
        return empty_figure(f"Mọi ô đều có < {MIN_N} khách")
    hot = stacked.loc[stacked.rate.idxmax()]
    fig.update_xaxes(title="Tuổi thiết bị", side="bottom")
    fig.update_yaxes(title="Thời gian sử dụng", autorange="reversed")
    return apply_style(fig, f"Điểm nóng: {hot.TenureBand} + máy {hot.EquipmentAgeBand} có tỷ lệ rời bỏ {fmt_pct(hot.rate)}",
                       f"Ô trống: nhóm có < {MIN_N} khách", height=420)


# ---------- C08–C12: hành vi dịch vụ ----------

def retention_funnel(df: pd.DataFrame) -> go.Figure:
    stages = [
        ("Tất cả khách", df),
        ("Đã gọi bộ phận giữ chân", df[df.MadeCallToRetentionTeam == "Yes"]),
        ("Đã nhận ưu đãi giữ chân", df[df.RetentionOfferAccepted == "Yes"]),
    ]
    names = [s for s, _ in stages]
    n = [len(d) for _, d in stages]
    rates = [baseline(d) if len(d) else 0 for _, d in stages]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.18,
                        subplot_titles=["Số khách", "Tỷ lệ rời bỏ"], specs=[[{"type": "funnel"}, {"type": "bar"}]])
    fig.add_trace(go.Funnel(y=names, x=n, marker=dict(color=[STAY, ACCENT, ACCENT]), textinfo="value",
                            texttemplate="%{value:,.0f}", hovertemplate="%{y}: %{x:,.0f} khách<extra></extra>"), 1, 1)
    fig.add_trace(go.Bar(y=names, x=rates, orientation="h", marker_color=[STAY, CHURN, CHURN],
                         text=[fmt_pct(r) for r in rates], textposition="outside",
                         hovertemplate="%{y}: %{x:.1%}<extra></extra>"), 1, 2)
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=2)
    fig.update_xaxes(tickformat=".0%", range=[0, max(rates) * 1.3 if max(rates) else 1], row=1, col=2)
    fig = apply_style(fig, f"Nhận ưu đãi giữ chân rồi vẫn có tỷ lệ rời bỏ {fmt_pct(rates[2])}: chương trình hiện tại không giữ được khách",
                      f"So với trung bình {fmt_pct(rates[0])} · khách đã liên hệ bộ phận giữ chân là nhóm báo động", height=360)
    fig.update_layout(showlegend=False)
    fig.update_annotations(font=dict(size=12, color=MUTED))
    return fig


def device_grouped(df: pd.DataFrame) -> go.Figure:
    rows = []
    for col, label in [("HandsetWebCapable", "Máy hỗ trợ mạng"), ("HandsetRefurbished", "Máy tân trang")]:
        t = rate_by(df[df[col].isin(["Yes", "No"])], col)
        t["feature"] = label
        t["value"] = t[col].map({"Yes": "Có", "No": "Không"})
        rows.append(t)
    table = pd.concat(rows, ignore_index=True)
    if table.empty:
        return empty_figure()
    fig = go.Figure()
    for value, color in [("Có", STAY), ("Không", ACCENT)]:
        part = table[table.value == value]
        fig.add_trace(go.Bar(x=part.feature, y=part.rate, name=value, marker_color=color,
                             text=[fmt_pct(r) for r in part.rate], textposition="outside",
                             customdata=_hover(part), hovertemplate=HOVER.replace("%{x}", f"%{{x}}: {value}")))
    # Máy không hỗ trợ mạng có tỷ lệ rời bỏ cao hơn → tô đỏ để nhấn mạnh
    web_no = table[(table.feature == "Máy hỗ trợ mạng") & (table.value == "Không")]
    web_yes = table[(table.feature == "Máy hỗ trợ mạng") & (table.value == "Có")]
    add_baseline(fig, baseline(df))
    fig.update_layout(barmode="group")
    fig.update_yaxes(tickformat=".0%", range=[0, table.rate.max() * 1.25], title="Tỷ lệ rời bỏ")
    title = "Đặc điểm thiết bị và tỷ lệ rời bỏ"
    if len(web_no) and len(web_yes):
        title = (f"Máy không hỗ trợ mạng có tỷ lệ rời bỏ {fmt_pct(web_no.rate.iloc[0])} so với "
                 f"{fmt_pct(web_yes.rate.iloc[0])} ở máy có mạng")
    return apply_style(fig, title, "Máy tân trang cũng có tỷ lệ rời bỏ cao hơn máy mới")


def overage_bubble(df: pd.DataFrame) -> go.Figure:
    data = df[~df.MinutesBand.isin(["Không rõ"]) & ~df.OverageBand.isin(["Không rõ"])]
    table = (data.groupby(["MinutesBand", "OverageBand"], observed=True)[TARGET]
             .agg(n="size", churners="sum").reset_index())
    table = table[table.n >= MIN_N]
    if table.empty:
        return empty_figure()
    table["rate"] = table.churners / table.n
    palette = {"0": NEUTRAL, "≤50": STAY, "50–200": ACCENT, ">200": CHURN}
    fig = go.Figure()
    for band in [b for b in palette if b in set(table.OverageBand.astype(str))]:
        part = table[table.OverageBand.astype(str) == band]
        fig.add_trace(go.Scatter(
            x=part.MinutesBand.astype(str), y=part.rate, mode="markers", name=f"Vượt gói {band} phút",
            marker=dict(size=part.n, sizemode="area", sizeref=2 * table.n.max() / 60 ** 2, sizemin=6,
                        color=palette[band], line=dict(color="white", width=1)),
            customdata=_hover(part), hovertemplate=HOVER.replace("%{x}", f"%{{x}} · vượt {band}")))
    add_baseline(fig, baseline(df))
    fig.update_yaxes(tickformat=".0%", title="Tỷ lệ rời bỏ", rangemode="tozero")
    fig.update_xaxes(title="Phút gọi / tháng", type="category",
                     categoryorder="array", categoryarray=["≤150 phút", "150–500 phút", ">500 phút"])
    over = rate_by(data, "OverageBand")
    top = over[over.OverageBand.astype(str) == ">200"]
    title = "Gói cước và tỷ lệ rời bỏ: dùng ít và vượt gói nhiều đều rủi ro"
    if len(top):
        title = f"Vượt gói > 200 phút có tỷ lệ rời bỏ {fmt_pct(top.rate.iloc[0])}; dùng ít (≤150 phút) cũng rủi ro"
    return apply_style(fig, title, "Kích thước bong bóng = số khách", height=400)


# ---------- C13–C15: địa lý ----------

def region_table(df: pd.DataFrame, min_n=800) -> pd.DataFrame:
    return rate_by(df[df.Region != "Unknown"], "Region", min_n=min_n)


def region_bar(df: pd.DataFrame, min_n=800) -> go.Figure:
    table = region_table(df, min_n)
    if len(table) < 2:
        table = region_table(df, MIN_N)
    if table.empty:
        return empty_figure()
    hi, lo = table.loc[table.rate.idxmax()], table.loc[table.rate.idxmin()]
    gap = 100 * (hi.rate - lo.rate)
    return rate_bar(df[df.Region.isin(table.Region)], "Region", "Vùng", horizontal=True, base=baseline(df),
                    title=f"Vùng {hi.Region} có tỷ lệ rời bỏ {fmt_pct(hi.rate)}, cao hơn {lo.Region} {fmt_ratio(gap)} điểm %")


def region_treemap(df: pd.DataFrame, top=12) -> go.Figure:
    churned = df[df[TARGET] == 1]
    if churned.empty:
        return empty_figure()
    top_regions = churned.Region.value_counts().head(top).index
    data = df.assign(Region=df.Region.where(df.Region.isin(top_regions), "Vùng khác"))
    table = (data.groupby(["Region", "TenureBand"], observed=True)[TARGET]
             .agg(n="size", churners="sum").reset_index())
    table = table[table.churners > 0]
    table["rate"] = table.churners / table.n
    fig = px.treemap(table, path=[px.Constant("Tất cả"), "Region", "TenureBand"], values="churners",
                     color="rate", color_continuous_scale="Reds", custom_data=["n", "rate"])
    fig.update_traces(hovertemplate="<b>%{label}</b><br>Khách rời bỏ: %{value:,.0f}<extra></extra>",
                      texttemplate="%{label}<br>%{value:,.0f}", root_color="white")
    fig.update_coloraxes(colorbar=dict(tickformat=".0%", title="Tỷ lệ rời bỏ"))
    lead = churned.Region.value_counts()
    lead = lead[lead.index != "Unknown"]
    share = lead.iloc[0] / len(churned) if len(lead) else 0
    return apply_style(fig, f"{lead.index[0]} mất nhiều khách nhất về số lượng ({fmt_pct(share)} tổng khách rời bỏ)",
                       f"Diện tích = số khách rời bỏ, màu = tỷ lệ rời bỏ · {top} vùng lớn nhất", height=460)


def region_pareto(df: pd.DataFrame) -> go.Figure:
    churned = df[(df[TARGET] == 1) & (df.Region != "Unknown")]
    if churned.empty:
        return empty_figure()
    counts = churned.Region.value_counts()
    cum = counts.cumsum() / counts.sum()
    k80 = int((cum < 0.8).sum() + 1)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=counts.index, y=counts.values, name="Số khách rời bỏ",
                         marker_color=[CHURN if i < k80 else NEUTRAL for i in range(len(counts))],
                         hovertemplate="%{x}: %{y:,.0f} khách rời bỏ<extra></extra>"), secondary_y=False)
    fig.add_trace(go.Scatter(x=cum.index, y=cum.values, name="% tích lũy", mode="lines",
                             line=dict(color=INK, width=2), hovertemplate="%{x}: %{y:.0%}<extra></extra>"),
                  secondary_y=True)
    fig.add_hline(y=0.8, line_dash="dot", line_color=MUTED, secondary_y=True)
    fig.update_yaxes(title="Số khách rời bỏ", secondary_y=False)
    fig.update_yaxes(title="% tích lũy", tickformat=".0%", range=[0, 1.05], secondary_y=True, showgrid=False)
    fig.update_xaxes(type="category", tickangle=-45)
    return apply_style(fig, f"{k80}/{len(counts)} vùng chiếm 80% số khách rời bỏ",
                       "Pareto: đỏ = nhóm vùng tạo nên 80% khách rời bỏ", height=400)


# ---------- C16–C18: khách hàng ----------

def credit_bar(df: pd.DataFrame) -> go.Figure:
    table = rate_by(df, "CreditRating", order=CREDIT_ORDER, min_n=MIN_N)
    if table.empty:
        return empty_figure()
    good = baseline(df[df.CreditRating.isin(CREDIT_ORDER[:3])])
    low = baseline(df[df.CreditRating.isin(CREDIT_ORDER[3:])])
    return rate_bar(df, "CreditRating", "Hạng tín dụng", order=CREDIT_ORDER,
                    title=f"Khách tín dụng tốt (hạng 1–3) có tỷ lệ rời bỏ {fmt_pct(good)}, cao hơn nhóm 4–7 ({fmt_pct(low)})")


def demographic_multiples(df: pd.DataFrame) -> go.Figure:
    dims = [("IncomeGroup", "Nhóm thu nhập"), ("Occupation", "Nghề nghiệp"),
            ("PrizmCode", "Khu vực sống"), ("MaritalStatus", "Hôn nhân")]
    fig = make_subplots(rows=1, cols=4, shared_yaxes=True, subplot_titles=[d[1] for d in dims], horizontal_spacing=0.03)
    base = baseline(df)
    spreads = []
    for i, (col, _) in enumerate(dims, start=1):
        t = rate_by(df, col, min_n=MIN_N)
        if col == "IncomeGroup":
            t = t.sort_values(col, key=lambda s: pd.to_numeric(s, errors="coerce"))
        else:
            t = t.sort_values("rate", ascending=False)
        if len(t):
            spreads.append(t.rate.max() - t.rate.min())
        fig.add_trace(go.Bar(x=t[col].astype(str).map(lambda value: display_value(col, value)), y=t.rate, marker_color=[CHURN if r > base else STAY for r in t.rate],
                             customdata=_hover(t), hovertemplate=HOVER, showlegend=False), row=1, col=i)
        fig.add_hline(y=base, line_dash="dash", line_color=MUTED, line_width=1, row=1, col=i)
        fig.update_xaxes(type="category", tickangle=-45, row=1, col=i)
    fig.update_yaxes(tickformat=".0%", rangemode="tozero")
    fig.update_yaxes(title="Tỷ lệ rời bỏ", row=1, col=1)
    spread = max(spreads) if spreads else 0
    fig = apply_style(fig, f"Nhân khẩu học chênh lệch nhỏ (tối đa {fmt_ratio(100 * spread)} điểm %): ưu tiên tín hiệu hành vi",
                      f"Cùng thang trục y · nhóm có ≥{MIN_N} khách · nét đứt = trung bình", height=420)
    fig.update_annotations(font=dict(size=12, color=MUTED))
    return fig


CORR_COLUMNS = [
    "ChurnFlag", "MonthsInService", "CurrentEquipmentDays", "MonthlyRevenue", "MonthlyMinutes",
    "TotalRecurringCharge", "OverageMinutes", "PercChangeMinutes", "DroppedBlockedCalls",
    "CustomerCareCalls", "ReceivedCalls", "PeakCallsInOut", "RetentionCalls", "HandsetModels", "AgeHH1",
]


C18_FEATURES = [
    "MonthsInService", "CurrentEquipmentDays", "HandsetWebCapable",
    "MonthlyRevenue", "TotalRecurringCharge", "CreditRating",
    "MonthlyMinutes", "PercChangeMinutes", "PercChangeRevenues", "PeakCallsInOut",
]
C18_COLUMNS = [TARGET, *C18_FEATURES]


def correlation_matrix(df: pd.DataFrame, columns=C18_COLUMNS) -> pd.DataFrame:
    """Chỉ mã hóa trên bản sao phục vụ C18, không thay đổi đầu vào mô hình."""
    data = df[columns].copy()
    if "HandsetWebCapable" in data:
        data["HandsetWebCapable"] = data.HandsetWebCapable.map({"Yes": 1, "No": 0, 1: 1, 0: 0})
    if "CreditRating" in data:
        rating = data.CreditRating.astype("string").str.extract(r"^([1-7])(?:-|$)", expand=False)
        data["CreditRating"] = pd.to_numeric(rating, errors="coerce")
    return data.corr()


def correlation_heatmap(df: pd.DataFrame, columns=C18_COLUMNS) -> go.Figure:
    corr = correlation_matrix(df, columns)
    labels = ["<br>".join(wrap(field_label(c), width=20)) for c in columns]
    text = [[f"{value:.2f}".replace(".", ",") if pd.notna(value) else "" for value in row]
            for row in corr.values]
    fig = go.Figure(go.Heatmap(z=corr.values, x=labels, y=labels, zmin=-1, zmax=1, zmid=0, colorscale="RdBu_r",
                               text=text, texttemplate="%{text}", textfont=dict(size=11),
                               hovertemplate="%{y} × %{x}: %{z:.2f}<extra></extra>"))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(tickangle=-45)
    with_target = corr[TARGET].drop(TARGET).dropna()
    if with_target.empty:
        return empty_figure("Lát cắt chỉ có một trạng thái rời bỏ: không tính được tương quan")
    top = with_target.abs().idxmax()
    r = f"{with_target[top]:.2f}".replace(".", ",")
    return apply_style(fig, f"Tương quan với rời bỏ cao nhất: {field_label(top).lower()} (r = {r})",
                       f"{len(columns) - 1} biến đầu vào + Rời bỏ · n = {fmt_n(len(df))} khách · hệ số Pearson", height=740)


# ---------- C19: mô hình rủi ro ----------

def decile_bar(df: pd.DataFrame) -> go.Figure:
    if "RiskDecile" not in df:
        return empty_figure("Chưa có điểm rủi ro: chạy work/pipeline/run_all.py")
    table = rate_by(df, "RiskDecile")
    table["RiskDecile"] = table.RiskDecile.astype(int)
    table = table.sort_values("RiskDecile")
    base = baseline(df)
    fig = go.Figure(go.Bar(x=table.RiskDecile.astype(str), y=table.rate,
                           marker_color=[CHURN if rate > base else STAY for rate in table.rate],
                           text=[fmt_pct(r) for r in table.rate], textposition="outside",
                           customdata=_hover(table), hovertemplate=HOVER.replace("%{x}", "Nhóm rủi ro %{x}")))
    add_baseline(fig, base)
    fig.update_xaxes(title="Nhóm rủi ro (1 = thấp nhất, 10 = cao nhất)", type="category")
    fig.update_yaxes(tickformat=".0%", range=[0, table.rate.max() * 1.2], title="Tỷ lệ rời bỏ thực tế")
    top = table[table.RiskDecile == table.RiskDecile.max()]
    title = "Tỷ lệ rời bỏ thực tế theo nhóm điểm rủi ro"
    if len(top) and base:
        title = f"Nhóm rủi ro cao nhất có tỷ lệ rời bỏ {fmt_pct(top.rate.iloc[0])}, gấp {fmt_ratio(top.rate.iloc[0] / base)} lần trung bình"
    return apply_style(fig, title, "Dự đoán ngoài mẫu từ CatBoost · đỏ = tỷ lệ rời bỏ cao hơn trung bình · mỗi nhóm khoảng 10% khách")


def gain_curve(df: pd.DataFrame) -> go.Figure:
    if "RiskScore" not in df or df[TARGET].sum() == 0:
        return empty_figure("Chưa có điểm rủi ro: chạy work/pipeline/run_all.py")
    ordered = df.sort_values("RiskScore", ascending=False)
    share = np.arange(1, len(ordered) + 1) / len(ordered)
    gain = ordered[TARGET].cumsum().to_numpy() / ordered[TARGET].sum()
    step = max(1, len(ordered) // 400)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=share[::step], y=gain[::step], name="Theo mô hình", line=dict(color=ACCENT, width=3),
                             hovertemplate="Liên hệ %{x:.0%} → bắt được %{y:.0%} khách rời bỏ<extra></extra>"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name="Chọn ngẫu nhiên", line=dict(color=MUTED, dash="dash")))
    i10 = int(0.1 * len(ordered)) - 1
    i20 = int(0.2 * len(ordered)) - 1
    for i in (i10, i20):
        fig.add_trace(go.Scatter(x=[share[i]], y=[gain[i]], mode="markers+text", showlegend=False,
                                 marker=dict(color=CHURN, size=9), text=[fmt_pct(gain[i], 0)],
                                 textposition="top left", hoverinfo="skip"))
    fig.update_xaxes(tickformat=".0%", title="% khách được liên hệ (xếp theo rủi ro)")
    fig.update_yaxes(tickformat=".0%", title="% khách rời bỏ bắt được")
    return apply_style(fig, f"Liên hệ 20% khách rủi ro nhất bắt được {fmt_pct(gain[i20], 0)} số khách sẽ rời bỏ",
                       f"10% khách rủi ro cao nhất: {fmt_pct(gain[i10], 0)} khách rời bỏ (hiệu quả so với chọn ngẫu nhiên {fmt_ratio(gain[i10] / 0.1)}×)", height=400)


# ---------- C20: phân khúc hành động ----------

def segment_bubble(df: pd.DataFrame) -> go.Figure:
    table = segment_summary(df)
    if table.empty:
        return empty_figure()
    base = baseline(df)
    colors = [NEUTRAL if s == "Ổn định" else (CHURN if r > base else STAY) for s, r in zip(table.ActionSegment, table.rate)]
    fig = go.Figure(go.Scatter(
        x=table.rate, y=table.avg_revenue, mode="markers+text", text=table.ActionSegment.astype(str),
        textposition="top center",
        marker=dict(size=table.n, sizemode="area", sizeref=2 * table.n.max() / 70 ** 2, sizemin=10,
                    color=colors, line=dict(color="white", width=1.5)),
        customdata=np.stack([table.n.map(fmt_n), table.rate.map(fmt_pct), table.revenue_at_risk.map(fmt_money)], -1),
        hovertemplate="<b>%{text}</b><br>Khách: %{customdata[0]}<br>Tỷ lệ rời bỏ: %{customdata[1]}"
                      "<br>Doanh thu/tháng bị đe dọa: %{customdata[2]}<extra></extra>"))
    add_baseline(fig, base, axis="x")
    fig.add_hline(y=df.MonthlyRevenue.mean(), line_dash="dot", line_color=NEUTRAL)
    fig.update_xaxes(tickformat=".0%", title="Tỷ lệ rời bỏ", range=[0, table.rate.max() * 1.25])
    fig.update_yaxes(title="Doanh thu TB / khách / tháng ($)", range=[table.avg_revenue.min() * 0.85, table.avg_revenue.max() * 1.15])
    act = table[table.ActionSegment != "Ổn định"]
    title = "Phân khúc ưu tiên giữ chân"
    if len(act):
        top = act.loc[act.revenue_at_risk.idxmax()]
        title = f"“{top.ActionSegment}” có doanh thu bị đe dọa lớn nhất: {fmt_money(top.revenue_at_risk)}/tháng"
    return apply_style(fig, title, "Bong bóng = số khách · phải đường đứt = tỷ lệ rời bỏ trên trung bình", height=440)


def segment_revenue_rank(df: pd.DataFrame) -> go.Figure:
    """Xếp hạng các nhóm cần can thiệp theo quy mô doanh thu bị đe dọa."""
    table = segment_summary(df)
    table = table[table.ActionSegment != "Ổn định"].sort_values("revenue_at_risk", ascending=False)
    if table.empty:
        return empty_figure("Không có nhóm cần can thiệp trong lát cắt này")
    labels = [f"{i}. {row.ActionSegment}<br>{fmt_n(row.n)} khách · {fmt_pct(row.rate)} rời bỏ"
              for i, row in enumerate(table.itertuples(), start=1)]
    custom = np.column_stack([table.n.map(fmt_n), table.churners.map(fmt_n),
                              table.rate.map(fmt_pct), table.revenue_at_risk.map(fmt_money)])
    fig = go.Figure(go.Bar(
        x=table.revenue_at_risk, y=labels, orientation="h", marker_color=ACCENT,
        text=table.revenue_at_risk.map(fmt_money), textposition="outside", cliponaxis=False,
        customdata=custom,
        hovertemplate="<b>%{y}</b><br>Số khách: %{customdata[0]}<br>Khách rời bỏ: %{customdata[1]}"
                      "<br>Tỷ lệ rời bỏ: %{customdata[2]}<br>Doanh thu tháng bị đe dọa: %{customdata[3]}<extra></extra>",
    ))
    top = table.iloc[0]
    fig.update_xaxes(title="Doanh thu tháng bị đe dọa ($)", tickformat=",.0f",
                     range=[0, max(float(table.revenue_at_risk.max()) * 1.35, 1)])
    fig.update_yaxes(title=None, autorange="reversed", categoryorder="array", categoryarray=labels)
    return apply_style(
        fig, f"Ưu tiên theo doanh thu: “{top.ActionSegment}” đứng đầu ({fmt_money(top.revenue_at_risk)}/tháng)",
        f"Chỉ các nhóm cần can thiệp · n = {fmt_n(table.n.sum())} khách · xếp theo doanh thu, chưa xét chi phí giữ chân",
        height=max(380, 80 * len(table) + 150),
    )


def segment_rate_rank(df: pd.DataFrame) -> go.Figure:
    """Biểu đồ chấm có thân nối, xếp hạng tỷ lệ rời bỏ và đối chiếu baseline."""
    table = segment_summary(df).sort_values("rate", ascending=False)
    if table.empty:
        return empty_figure()
    base = baseline(df)
    labels = [f"{row.ActionSegment}<br>{fmt_n(row.n)} khách" for row in table.itertuples()]
    stem_x, stem_y = [], []
    for label, rate in zip(labels, table.rate):
        stem_x.extend([0, rate, None])
        stem_y.extend([label, label, None])
    fig = go.Figure(go.Scatter(x=stem_x, y=stem_y, mode="lines", line=dict(color=NEUTRAL, width=3),
                               hoverinfo="skip", showlegend=False))
    colors = [NEUTRAL if segment == "Ổn định" else (CHURN if rate > base else STAY)
              for segment, rate in zip(table.ActionSegment, table.rate)]
    fig.add_trace(go.Scatter(
        x=table.rate, y=labels, mode="markers+text", marker=dict(size=14, color=colors),
        text=table.rate.map(fmt_pct), textposition="middle right", showlegend=False,
        customdata=_hover(table), hovertemplate=HOVER_H,
    ))
    add_baseline(fig, base, axis="x")
    fig.update_xaxes(title="Tỷ lệ khách thực tế rời bỏ", tickformat=".0%",
                     range=[0, max(float(table.rate.max()), base, 0.01) * 1.25])
    fig.update_yaxes(title=None, autorange="reversed", categoryorder="array", categoryarray=labels)
    top = table.iloc[0]
    return apply_style(
        fig, f"Ưu tiên theo tỷ lệ rời bỏ: “{top.ActionSegment}” cao nhất ({fmt_pct(top.rate)})",
        f"n = {fmt_n(table.n.sum())} khách · đường đứt = trung bình {fmt_pct(base)} · nhóm ổn định để đối chiếu",
        height=max(400, 72 * len(table) + 140),
    )


# ---------- danh mục C01–C20 (tên file PNG, hàm dựng) ----------

def _cats(df, col):
    return list(df[col].cat.categories) if hasattr(df[col], "cat") else None


FIGURES = {
    "C01_ty_le_churn": churn_donut,
    "C02_du_lieu_thieu": lambda df: missing_bar(df, CORR_COLUMNS[1:]),
    "C03_phan_bo_tuoi_may": lambda df: distribution_overlay(df, "CurrentEquipmentDays", "Tuổi thiết bị"),
    "C04_box_plot": lambda df: box_by_churn(df, {"CurrentEquipmentDays": "Tuổi thiết bị (ngày)",
                                                "MonthlyRevenue": "Doanh thu/tháng ($)",
                                                "OverageMinutes": "Phút vượt gói"}),
    "C05_vong_doi": tenure_line,
    "C05b_so_khach": tenure_stacked,
    "C06_tuoi_thiet_bi": lambda df: rate_bar(df, "EquipmentAgeBand", "Máy", order=_cats(df, "EquipmentAgeBand")),
    "C07_heatmap_tenure_thiet_bi": tenure_equipment_heatmap,
    "C08_retention_funnel": retention_funnel,
    "C09_dac_diem_thiet_bi": device_grouped,
    "C10_goi_cskh": lambda df: rate_bar(df, "CareCallsBand", "Số cuộc gọi CSKH", order=_cats(df, "CareCallsBand")),
    "C11_cuoc_goi_rot": lambda df: rate_bar(df, "DroppedBand", "Số cuộc gọi rớt", order=_cats(df, "DroppedBand")),
    "C12_goi_cuoc": overage_bubble,
    "C13_vung": region_bar,
    "C14_treemap_vung": region_treemap,
    "C15_pareto_vung": region_pareto,
    "C16_tin_dung": credit_bar,
    "C17_nhan_khau_hoc": demographic_multiples,
    "C18_tuong_quan": correlation_heatmap,
    "C19a_risk_decile": decile_bar,
    "C19b_gain_curve": gain_curve,
    "C20_phan_khuc": segment_bubble,
    "C20b_uu_tien_doanh_thu": segment_revenue_rank,
    "C20c_uu_tien_ty_le": segment_rate_rank,
}
'''
c = ModuleType("notebook4_full_charts")
exec(compile(chart_source, "<notebook4_full_charts>", "exec"), c.__dict__)


In [ ]:
# Chu?n b? ??y ?? c?c tr??ng EDA theo c?ng commit; ch? ??c d? li?u ngu?n v? ?i?m ?? c?.
preparation_source = r'''"""Chuẩn bị bảng phân tích cho phần trực quan hóa: python work/viz/prepare_viz_data.py"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
RAW = ROOT / "data" / "cell2celltrain.csv"
BUILD = ROOT / "work" / "build"
MART = BUILD / "customer_mart.parquet"
OUT = BUILD / "viz_customers.parquet"

NUMERIC = [
    "MonthlyRevenue", "MonthlyMinutes", "TotalRecurringCharge", "DirectorAssistedCalls", "OverageMinutes",
    "RoamingCalls", "PercChangeMinutes", "PercChangeRevenues", "DroppedBlockedCalls", "UnansweredCalls",
    "CustomerCareCalls", "ThreewayCalls", "ReceivedCalls", "OutboundCalls", "InboundCalls", "PeakCallsInOut",
    "OffPeakCallsInOut", "CallWaitingCalls", "MonthsInService", "UniqueSubs", "ActiveSubs", "Handsets",
    "HandsetModels", "CurrentEquipmentDays", "AgeHH1", "RetentionCalls", "RetentionOffersAccepted",
    "ReferralsMadeBySubscriber", "AdjustmentsToCreditRating",
]
CATEGORICAL = [
    "ServiceArea", "HandsetWebCapable", "HandsetRefurbished", "HandsetPrice", "MadeCallToRetentionTeam",
    "CreditRating", "IncomeGroup", "PrizmCode", "Occupation", "MaritalStatus",
]

# Các mức thay đổi có thể âm hợp lệ; các đại lượng còn lại không được âm.
SIGNED_COLUMNS = {"PercChangeMinutes", "PercChangeRevenues"}
NON_NEGATIVE_COLUMNS = [column for column in NUMERIC if column not in SIGNED_COLUMNS]


def remove_invalid_negative_rows(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Loại dòng có số âm bất thường, kèm số trường hợp theo từng cột để đối soát."""
    negative = raw[NON_NEGATIVE_COLUMNS].lt(0)
    counts = negative.sum().loc[lambda values: values.gt(0)].astype(int)
    return raw.loc[~negative.any(axis=1)].copy(), counts

# Ngưỡng band công khai; pd.cut đóng bên phải.
BANDS = {
    "TenureBand": ("MonthsInService", [0, 6, 12, 18, 24, 36, np.inf],
                   ["0–6 tháng", "7–12 tháng", "13–18 tháng", "19–24 tháng", "25–36 tháng", ">36 tháng"]),
    "EquipmentAgeBand": ("CurrentEquipmentDays", [-0.1, 180, 365, 730, np.inf],
                         ["≤180 ngày", "181–365 ngày", "366–730 ngày", ">730 ngày"]),
    "CareCallsBand": ("CustomerCareCalls", [-0.1, 0, 1, 3, 10, np.inf], ["0", "≤1", "1–3", "3–10", ">10"]),
    "DroppedBand": ("DroppedBlockedCalls", [-0.1, 0, 5, 15, 40, np.inf], ["0", "≤5", "5–15", "15–40", ">40"]),
    "OverageBand": ("OverageMinutes", [-0.1, 0, 50, 200, np.inf], ["0", "≤50", "50–200", ">200"]),
    "MinutesBand": ("MonthlyMinutes", [-0.1, 150, 500, np.inf], ["≤150 phút", "150–500 phút", ">500 phút"]),
}

def add_bands(df: pd.DataFrame) -> pd.DataFrame:
    for band, (col, bins, labels) in BANDS.items():
        values = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True).astype("object")
        values[df[col].lt(0)] = "Bất thường"
        values[df[col].isna()] = "Không rõ"
        order = labels + [x for x in ["Bất thường", "Không rõ"] if x in set(values)]
        df[band] = pd.Categorical(values, categories=order, ordered=True)
    return df


SEGMENTS = ["Đã báo động", "Hết hợp đồng", "Máy cũ", "Rủi ro cao (mô hình)", "Ổn định"]


def add_segment(df: pd.DataFrame) -> pd.DataFrame:
    # Rule có thứ tự ưu tiên: điều kiện đầu tiên khớp sẽ quyết định segment.
    # Tháng 10–13 bao quanh đỉnh churn tháng 11–12 (hết hợp đồng 1 năm).
    conditions = [
        df.MadeCallToRetentionTeam.eq("Yes"),
        df.MonthsInService.between(10, 13),
        df.CurrentEquipmentDays.gt(730) | (df.CurrentEquipmentDays.ge(365) & df.HandsetWebCapable.eq("No")),
    ]
    labels = SEGMENTS[:3]
    if "RiskDecile" in df:
        conditions.append(df.RiskDecile.ge(8))
        labels.append(SEGMENTS[3])
    segment = np.select(conditions, labels, default=SEGMENTS[-1])
    df["ActionSegment"] = pd.Categorical(segment, categories=[*labels, SEGMENTS[-1]], ordered=True)
    return df


def prepare_data(raw: pd.DataFrame) -> pd.DataFrame:
    """Chuẩn hóa bảng đã kiểm tra; không đọc lại dữ liệu nguồn hoặc huấn luyện mô hình."""
    if raw[NON_NEGATIVE_COLUMNS].lt(0).any().any():
        raise ValueError("Dữ liệu còn số âm bất thường. Hãy chạy remove_invalid_negative_rows trước.")
    df = raw[["CustomerID", "Churn", *NUMERIC, *CATEGORICAL]].copy()
    if not df.Churn.isin(["Yes", "No"]).all():
        raise SystemExit("Cột Churn có giá trị khác Yes/No.")
    df["ChurnFlag"] = df.Churn.eq("Yes").astype(int)
    for col in CATEGORICAL:
        df[col] = df[col].fillna("Unknown").astype(str)
    df["Region"] = df.ServiceArea.where(df.ServiceArea.ne("Unknown"), None).str[:3].fillna("Unknown")
    df["RetentionOfferAccepted"] = np.where(df.RetentionOffersAccepted.gt(0), "Yes", "No")
    df = add_bands(df)

    if MART.exists():
        mart = pd.read_parquet(MART, columns=["CustomerID", "RiskScore", "RiskDecile"])
        mart["CustomerID"] = mart.CustomerID.astype("int64")
        df = df.merge(mart, on="CustomerID", how="left", validate="one_to_one")
    else:
        print(f"Không thấy {MART.name}: bỏ qua RiskScore/RiskDecile (chạy work/pipeline/run_all.py nếu cần).")
    return add_segment(df)


def build() -> pd.DataFrame:
    raw = pd.read_csv(RAW, na_values=["NA"])
    cleaned, _ = remove_invalid_negative_rows(raw)
    return prepare_data(cleaned)


def fmt_n(value) -> str:
    return f"{int(value):,}".replace(",", ".")


def fmt_pct(value) -> str:
    return f"{100 * value:.2f}%".replace(".", ",")


def main() -> None:
    raw = pd.read_csv(RAW, na_values=["NA"])
    cleaned, negative_counts = remove_invalid_negative_rows(raw)
    df = prepare_data(cleaned)
    BUILD.mkdir(parents=True, exist_ok=True)
    df.to_parquet(OUT, index=False)

    print(f"Đã ghi {OUT.relative_to(ROOT)}: {fmt_n(len(df))} dòng, {df.shape[1]} cột")
    print(f"Đã loại {fmt_n(len(raw) - len(cleaned))} dòng có số âm bất thường; còn {fmt_n(len(cleaned))} khách.")
    print("Số âm theo cột (một dòng có thể xuất hiện ở nhiều cột):")
    print(negative_counts.to_string() if len(negative_counts) else "Không có.")
    print(f"Rời bỏ: {fmt_n(df.ChurnFlag.sum())} khách ({fmt_pct(df.ChurnFlag.mean())})")
    missing = df[NUMERIC].isna().mean().loc[lambda s: s.gt(0)].sort_values(ascending=False)
    print("\n% missing (cột numeric có missing):")
    print((100 * missing).round(2).to_string())
    print("\nSố khách theo ActionSegment:")
    print(df.groupby("ActionSegment", observed=False).ChurnFlag.agg(n="size", churn_rate="mean").round(3).to_string())
    if "RiskDecile" in df:
        print(f"\nRiskDecile có cho {fmt_n(df.RiskDecile.notna().sum())} khách.")


if __name__ == "__main__":
    main()
'''
preparation = ModuleType("notebook4_full_preparation")
preparation.__file__ = str(ROOT / "work" / "viz" / "prepare_viz_data.py")
exec(compile(preparation_source, "<notebook4_full_preparation>", "exec"), preparation.__dict__)


## 2. Tổng quan dữ liệu

Nguồn: `data/cell2celltrain.csv`, mỗi dòng là một khách hàng tại một thời điểm quan sát (dữ liệu tại một thời điểm), gồm 58 cột.
Biến mục tiêu **Trạng thái rời bỏ** = Rời bỏ/Ở lại.

In [ ]:
raw = pd.read_csv(ROOT / "data" / "cell2celltrain.csv", na_values=["NA"])
print(f"Dữ liệu nguồn: {c.fmt_n(raw.shape[0])} dòng × {raw.shape[1]} cột · Mã khách hàng trùng: {raw.CustomerID.duplicated().sum()}")
data_types = raw.dtypes.astype(str).map({"int64": "Số nguyên", "float64": "Số thực", "object": "Văn bản"})
display(data_types.value_counts().rename_axis("Kiểu dữ liệu").to_frame("Số cột"))
c.display_table(raw.head())

In [ ]:
c.display_table(raw.describe().T[["count", "mean", "50%", "min", "max"]].round(1).head(25))

## 3. Xử lý dữ liệu

Dùng cùng quy tắc trong `prepare_viz_data.py`: nhận diện giá trị thiếu; loại số âm bất thường ở các đại lượng không thể âm; giữ số âm ở hai mức thay đổi; tạo band và ghép dự đoán ngoài mẫu mới nhất theo mã khách.

Bảng EDA có thể giữ các cột mô tả bổ sung. Bộ đầu vào CatBoost chỉ gồm 10 biến đã chọn; các band và biến mô tả khác không được đưa vào mô hình.

In [ ]:
cleaned_raw, negative_counts = preparation.remove_invalid_negative_rows(raw)
removed_count = len(raw) - len(cleaned_raw)
metrics_path = ROOT / "work" / "build" / "model_metrics.json"
model_info = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.exists() else {}
if preparation.MART.exists() and model_info.get("features") != c.C18_FEATURES:
    raise ValueError("Điểm mô hình chưa dùng đúng bộ 10 biến. Hãy chạy notebook 03 trước.")
df = preparation.prepare_data(cleaned_raw)
k = c.kpis(df)
print(f"Đã loại {c.fmt_n(removed_count)} dòng có số âm bất thường từ {c.fmt_n(len(raw))} khách nguồn.")
print(f"Bảng EDA: {c.fmt_n(len(df))} khách · {c.fmt_n(k['churners'])} rời bỏ · {c.fmt_pct(k['rate'], 2)}")
print(f"Mô hình: {len(model_info.get('features', []))} biến · {model_info.get('model_version', 'chưa có điểm')}")
c.display_table(df[["TenureBand", "EquipmentAgeBand", "CareCallsBand", "DroppedBand", "OverageBand", "MinutesBand", "ActionSegment"]].describe())

### C02 — Dữ liệu thiếu ở đâu?

In [ ]:
show("C02_du_lieu_thieu")

In [ ]:
explain('C02_du_lieu_thieu')

## 4. Tổng quan tình trạng rời bỏ

### C01 — Tình trạng rời bỏ phổ biến đến đâu?

In [ ]:
show("C01_ty_le_churn")

In [ ]:
explain('C01_ty_le_churn')

### C03 — Phân bố tuổi thiết bị khác nhau thế nào giữa hai nhóm?

In [ ]:
show("C03_phan_bo_tuoi_may")

In [ ]:
explain('C03_phan_bo_tuoi_may')

### C04 — Giá trị ngoại lệ và độ phân tán ở các biến chính?

In [ ]:
show("C04_box_plot")

In [ ]:
explain('C04_box_plot')

## 5. Vòng đời và thiết bị

### C05 — Tỷ lệ rời bỏ thay đổi thế nào theo thời gian sử dụng?

In [ ]:
show("C05_vong_doi")
show("C05b_so_khach")

In [ ]:
explain('C05_vong_doi')

### C06 — Tuổi thiết bị ảnh hưởng ra sao?

In [ ]:
show("C06_tuoi_thiet_bi")

In [ ]:
explain('C06_tuoi_thiet_bi')

### C07 — Thời gian sử dụng và tuổi máy kết hợp thế nào?

In [ ]:
show("C07_heatmap_tenure_thiet_bi")

In [ ]:
explain('C07_heatmap_tenure_thiet_bi')

### C09 — Máy không hỗ trợ mạng hoặc máy tân trang có rời bỏ cao hơn?

In [ ]:
show("C09_dac_diem_thiet_bi")

In [ ]:
explain('C09_dac_diem_thiet_bi')

## 6. Hành vi sử dụng dịch vụ

### C08 — Chương trình giữ chân hiện tại có hiệu quả không?

In [ ]:
show("C08_retention_funnel")

In [ ]:
explain('C08_retention_funnel')

### C10 — Gọi chăm sóc khách hàng nhiều có phải dấu hiệu bất mãn?

In [ ]:
show("C10_goi_cskh")

In [ ]:
explain('C10_goi_cskh')

### C11 — Chất lượng mạng (cuộc gọi rớt/bị chặn) có gây rời bỏ?

In [ ]:
show("C11_cuoc_goi_rot")

In [ ]:
explain('C11_cuoc_goi_rot')

### C12 — Gói cước có phù hợp với nhu cầu?

In [ ]:
show("C12_goi_cuoc")

In [ ]:
explain('C12_goi_cuoc')

## 7. Địa lý

### C13 — Vùng nào rời bỏ cao nhất?

In [ ]:
show("C13_vung")

In [ ]:
explain('C13_vung')

### C14 — Khách rời bỏ tập trung ở đâu về số lượng?

In [ ]:
show("C14_treemap_vung")

In [ ]:
explain('C14_treemap_vung')

### C15 — Bao nhiêu vùng chiếm 80% số khách rời bỏ?

In [ ]:
show("C15_pareto_vung")

In [ ]:
explain('C15_pareto_vung')

## 8. Đặc điểm khách hàng

### C16 — Hạng tín dụng liên quan thế nào đến rời bỏ?

In [ ]:
show("C16_tin_dung")

In [ ]:
explain('C16_tin_dung')

### C17 — Thu nhập, nghề nghiệp, khu vực sống, hôn nhân có quan trọng?

In [ ]:
show("C17_nhan_khau_hoc")

In [ ]:
explain('C17_nhan_khau_hoc')

### C18 — 10 biến chính tương quan với rời bỏ và với nhau như thế nào?

Chỉ xét **10 biến đầu vào của mô hình hiện tại**: thời gian sử dụng, tuổi thiết bị, hỗ trợ web, doanh thu tháng, phí định kỳ, hạng tín dụng, phút gọi mỗi tháng, thay đổi phút gọi, thay đổi doanh thu và cuộc gọi giờ cao điểm. Thêm nhãn **Rời bỏ** để đối chiếu, tạo ma trận 11 × 11.

**Cách tính:** tương quan Pearson; hỗ trợ web Có = 1, Không = 0; hạng tín dụng từ 1 (cao nhất) đến 7 (thấp nhất). Giá trị thiếu/không rõ được bỏ theo từng cặp biến. Mã hóa chỉ phục vụ biểu đồ này; mô hình hiện tại cũng chỉ dùng 10 biến này.

In [ ]:
show("C18_tuong_quan")

In [ ]:
c18_corr = c.correlation_matrix(df)
c18_target = c18_corr[c.TARGET].drop(c.TARGET).dropna()
if not c18_target.empty:
    c18_top = c18_target.abs().idxmax()
    c18_r = f"{c18_target[c18_top]:.2f}".replace(".", ",")
    display(Markdown(
        f"**Nhận xét:** trong **{len(c.C18_FEATURES)} biến chính**, **{c.field_label(c18_top)}** có tương quan tuyến tính "
        f"lớn nhất về độ lớn với rời bỏ (**r = {c18_r}**). "
        "Hệ số mô tả liên hệ tuyến tính, không phải mức độ quan trọng của biến trong CatBoost. "
        "Với hạng tín dụng, số lớn hơn thể hiện hạng thấp hơn."
    ))
else:
    display(Markdown("**Nhận xét:** lát cắt chưa đủ biến thiên để tính tương quan với rời bỏ."))

## 9. Ai cần ưu tiên — mô hình rủi ro và phân khúc hành động

Điểm rủi ro lấy từ notebook 03, CatBoost chỉ dùng đúng 10 biến đầu vào và đánh giá ngoài mẫu theo ba fold. Ở đây tái sử dụng điểm của lần chạy mới, không huấn luyện mô hình trong notebook EDA. Số khách trong decile, gain, lift và phân khúc được tính lại từ điểm đó.

### C19a — Điểm rủi ro có phân tầng được khách hàng?

Mỗi nhóm gồm khoảng 10% khách, xếp từ rủi ro thấp đến cao. Cột đỏ có tỷ lệ rời bỏ thực tế vượt đường trung bình của tập khách đang phân tích; cột xanh xám có tỷ lệ bằng hoặc thấp hơn mức này.

In [ ]:
show("C19a_risk_decile")

In [ ]:
risk_groups = c.rate_by(df, "RiskDecile").sort_values("RiskDecile")
display(c.display_table(risk_groups.assign(rate=risk_groups.rate.map(c.fmt_pct))))
highest = risk_groups.iloc[-1]
above_average = risk_groups.loc[risk_groups.rate > c.baseline(df), "RiskDecile"].astype(int)
group_names = ", ".join(above_average.astype(str))
display(Markdown(
    f"**Nhận xét:** nhóm rủi ro {int(highest.RiskDecile)} có tỷ lệ rời bỏ "
    f"**{c.fmt_pct(highest.rate)}**, gấp {c.fmt_ratio(highest.rate / c.baseline(df))} lần trung bình. "
    f"Các nhóm **{group_names}** vượt mức trung bình **{c.fmt_pct(c.baseline(df), 2)}**, nên được tô đỏ. "
    "Màu sắc thể hiện tỷ lệ quan sát thực tế; điểm mô hình dùng để xếp thứ tự các nhóm."
))

### C19b — Với ngân sách hữu hạn, liên hệ bao nhiêu khách là đủ?

In [ ]:
show("C19b_gain_curve")

In [ ]:
ordered = df.sort_values("RiskScore", ascending=False)
total_churners = int(ordered.ChurnFlag.sum())
n10 = max(1, (len(ordered) + 9) // 10)
n20 = max(1, (len(ordered) + 4) // 5)
gain10 = ordered.head(n10).ChurnFlag.sum() / total_churners
gain20 = ordered.head(n20).ChurnFlag.sum() / total_churners
lift10 = ordered.head(n10).ChurnFlag.mean() / c.baseline(df)
display(Markdown(
    f"**Nhận xét:** liên hệ 10% khách có rủi ro cao nhất tiếp cận được **{c.fmt_pct(gain10, 0)}** "
    f"số khách thực tế rời bỏ, hiệu quả gấp **{c.fmt_ratio(lift10)} lần** chọn ngẫu nhiên. "
    f"Liên hệ 20% khách có rủi ro cao nhất tiếp cận được **{c.fmt_pct(gain20, 0)}** số khách rời bỏ."
))

**Quy tắc phân khúc hành động** (**Phân khúc hành động**, xét theo thứ tự, khách khớp quy tắc đầu tiên sẽ thuộc nhóm đó):

| # | Phân khúc | Điều kiện | Căn cứ |
|---|---|---|---|
| 1 | Đã báo động | Đã gọi bộ phận giữ chân | C08 |
| 2 | Hết hợp đồng | Tháng sử dụng 10–13 | C05 |
| 3 | Máy cũ | Máy > 730 ngày, hoặc ≥ 365 ngày và không hỗ trợ mạng | C06, C07, C09 |
| 4 | Rủi ro cao (mô hình) | Còn lại, nhóm rủi ro 8–10 | C19 |
| 5 | Ổn định | Còn lại | |

Trong bản nháp có thử nhóm "Sai gói" (vượt gói > 100 phút) và "Im lặng" (không gọi CSKH và < 150 phút). Sau khi đã tách 3 nhóm đầu,
hai nhóm này rời bỏ **thấp hơn** trung bình (24,3% và 25,1%) nên đã bỏ: tín hiệu của chúng đã nằm trong các nhóm trên.

In [ ]:
seg = c.segment_summary(df)
c.display_table(seg.assign(rate=seg.rate.map(c.fmt_pct), revenue_at_risk=seg.revenue_at_risk.map(c.fmt_money),
                           avg_revenue=seg.avg_revenue.round(1)).drop(columns="revenue"))

### C20 — Phân khúc nào cần ưu tiên, theo rủi ro và giá trị?

In [ ]:
show("C20_phan_khuc")

In [ ]:
intervention = seg[seg.ActionSegment != "Ổn định"]
stable = seg[seg.ActionSegment == "Ổn định"]
largest = intervention.loc[intervention.revenue_at_risk.idxmax()]
display(Markdown(
    f"**Nhận xét:** {len(intervention)} nhóm cần can thiệp gồm **{c.fmt_n(intervention.n.sum())} khách** "
    f"({c.fmt_pct(intervention.n.sum() / len(df), 0)} mẫu). Nhóm ổn định có tỷ lệ rời bỏ "
    f"**{c.fmt_pct(stable.rate.iloc[0])}**. Nhóm **{largest.ActionSegment}** có doanh thu tháng bị đe dọa "
    f"lớn nhất: **{c.fmt_money(largest.revenue_at_risk)}**."
))

### C20b — Nên ưu tiên nhóm nào theo quy mô doanh thu tháng bị đe dọa?

Thanh ngang xếp giảm dần theo **doanh thu tháng bị đe dọa** của các nhóm cần can thiệp. Nhãn cạnh mỗi nhóm ghi số khách và tỷ lệ rời bỏ; nhóm ổn định không nằm trong bảng xếp hạng này.

In [ ]:
# Nạp hai hàm mới nếu kernel đã import charts trước khi file được cập nhật.
show("C20b_uu_tien_doanh_thu")

revenue_priority = c.segment_summary(df)
revenue_priority = revenue_priority[revenue_priority.ActionSegment != "Ổn định"].sort_values("revenue_at_risk", ascending=False)
if not revenue_priority.empty:
    revenue_leader = revenue_priority.iloc[0]
    display(Markdown(
        f"**Nhận xét:** trong các nhóm cần can thiệp, **{revenue_leader.ActionSegment}** đứng đầu theo quy mô doanh thu tháng bị đe dọa: "
        f"**{c.fmt_money(revenue_leader.revenue_at_risk)}**, gồm **{c.fmt_n(revenue_leader.n)} khách**, tỷ lệ rời bỏ **{c.fmt_pct(revenue_leader.rate)}**. "
        "Chỉ số = tổng doanh thu tháng của nhóm × tỷ lệ rời bỏ; đây là quy mô tham khảo, chưa phải doanh thu chắc chắn giữ được."
    ))

### C20c — Nhóm nào có tỷ lệ rời bỏ cao nhất?

Biểu đồ chấm có thân nối (**lollipop**) xếp giảm dần theo **tỷ lệ rời bỏ thực tế**. Chấm càng sang phải, tỷ lệ càng cao; đường đứt là trung bình của toàn bộ mẫu đang xét. Nhóm ổn định được giữ lại làm đối chiếu.

In [ ]:
show("C20c_uu_tien_ty_le")

rate_priority = c.segment_summary(df).sort_values("rate", ascending=False)
if not rate_priority.empty:
    rate_leader = rate_priority.iloc[0]
    display(Markdown(
        f"**Nhận xét:** nhóm **{rate_leader.ActionSegment}** có tỷ lệ rời bỏ cao nhất: **{c.fmt_pct(rate_leader.rate)}**, "
        f"so với trung bình **{c.fmt_pct(c.baseline(df))}**, trên **{c.fmt_n(rate_leader.n)} khách**. "
        "Cách xếp này cho thấy nhóm rời bỏ nhiều nhất theo tỷ lệ; thứ tự có thể khác xếp hạng theo doanh thu ở C20b."
    ))

In [ ]:
policies = {
    "Đã báo động": ("Thiết kế lại ưu đãi; chuyên viên cấp cao xử lý trong 24 giờ; ghi nhận lý do rời bỏ", "Tỷ lệ rời bỏ sau 60 ngày; tỷ lệ nhận ưu đãi"),
    "Hết hợp đồng": ("Chăm sóc từ tháng 10; ưu đãi gia hạn 12 tháng trước tháng 11", "Tỷ lệ gia hạn; tỷ lệ rời bỏ tháng 11–13"),
    "Máy cũ": ("Đổi máy trả góp hoặc trợ giá gắn cam kết 12 tháng; ưu tiên máy không hỗ trợ mạng", "Tỷ lệ nhận đổi máy; tỷ lệ rời bỏ sau 90 ngày; chi phí giữ một khách"),
    "Rủi ro cao (mô hình)": ("Khảo sát hài lòng; tư vấn gói phù hợp; ưu đãi cá nhân hóa", "Tỷ lệ rời bỏ sau 90 ngày; doanh thu giữ được"),
    "Ổn định": ("Duy trì chất lượng và chương trình khách hàng thân thiết", "Theo dõi định kỳ"),
}
rows = ["| Phân khúc | Số khách | Tỷ lệ rời bỏ | Chính sách đề xuất | Chỉ số đo hiệu quả |",
        "|---|---:|---:|---|---|"]
for row in seg.itertuples():
    action, metric = policies[str(row.ActionSegment)]
    rows.append(f"| {row.ActionSegment} | {c.fmt_n(row.n)} | {c.fmt_pct(row.rate)} | {action} | {metric} |")
display(Markdown("## 10. Đề xuất chính sách giữ chân\n\n" + "\n".join(rows) +
                 "\n\nƯu tiên triển khai tại các vùng có nhiều khách rời bỏ và tỷ lệ rời bỏ cao. "
                 "Cần thử nghiệm đối chứng ngẫu nhiên trước khi triển khai rộng."))

In [ ]:
display(Markdown(
    "## 11. Tổng kết từ kết quả lần chạy này\n\n"
    f"- Tập nguồn **{c.fmt_n(len(raw))} khách**, loại **{c.fmt_n(removed_count)} dòng** có số âm bất thường, còn **{c.fmt_n(len(df))} khách**.\n"
    f"- Khách rời bỏ **{c.fmt_n(k['churners'])}**, tỷ lệ **{c.fmt_pct(k['rate'], 2)}**; doanh thu tháng của nhóm này **{c.fmt_money(k['lost_revenue'])}**.\n"
    f"- Mô hình dùng **{len(model_info.get('features', []))} biến**, ROC-AUC toàn mẫu ngoài mẫu **{model_info['roc_auc']:.4f}**, AP **{model_info['pr_auc']:.4f}**.\n"
    f"- Liên hệ 10% khách xếp rủi ro cao nhất trong tập EDA tiếp cận **{c.fmt_pct(gain10)}** khách rời bỏ, lift **{c.fmt_ratio(lift10)}×**.\n"
    f"- Các nhóm cần can thiệp gồm **{c.fmt_n(intervention.n.sum())} khách**; nhóm có doanh thu tháng bị đe dọa lớn nhất là **{largest.ActionSegment}** (**{c.fmt_money(largest.revenue_at_risk)}**).\n\n"
    "Các quan hệ là quan sát trên dữ liệu tại một thời điểm; chính sách cần thử nghiệm đối chứng trước khi triển khai rộng."
))

## 12. T?ng h?p k?t qu? trong notebook

B?ng d??i ??y ???c t?nh t? d? li?u c?a l?n ch?y n?y; kh?ng ghi ?? b?o c?o ho?c d? li?u dashboard.

In [ ]:
report_lines = [
    "# Insight và chính sách giữ chân khách hàng",
    "",
    f"Nguồn: {c.fmt_n(len(raw))} khách; sau khi loại {c.fmt_n(removed_count)} dòng có số âm bất thường, tập EDA còn **{c.fmt_n(len(df))} khách**, **{c.fmt_n(k['churners'])} khách rời bỏ ({c.fmt_pct(k['rate'], 2)})**.",
    "",
    f"Mô hình **{model_info['model_version']}** chỉ dùng **{len(model_info['features'])} biến đầu vào**. Các biến khảo sát bổ sung trong EDA không phải predictor của mô hình.",
    "",
    "## Các phát hiện từ dữ liệu hiện hành",
    "",
    "| Biểu đồ | Nhận xét tính từ dữ liệu |",
    "|---|---|",
]
for key in ["C01_ty_le_churn", "C05_vong_doi", "C06_tuoi_thiet_bi", "C07_heatmap_tenure_thiet_bi",
            "C08_retention_funnel", "C10_goi_cskh", "C11_cuoc_goi_rot", "C13_vung", "C18_tuong_quan", "C19b_gain_curve"]:
    title = dashboard_figures[key].layout.title.text.split("<br>")[0].replace("<b>", "").replace("</b>", "")
    report_lines.append(f"| [{key.split('_')[0]}](reports/figures/{key}.png) | {title} |")
report_lines += [
    "",
    "## Kết quả mô hình 10 biến",
    "",
    f"Đánh giá ngoài mẫu trên {c.fmt_n(len(raw))} khách nguồn: ROC-AUC **{model_info['roc_auc']:.4f}**, AP **{model_info['pr_auc']:.4f}**, lift top 10% **{c.fmt_ratio(model_info['capacity_10']['lift'])}×**, recall top 20% **{c.fmt_pct(model_info['capacity_20']['recall'])}**.",
    "",
    f"Trong tập EDA sau xử lý: lift top 10% **{c.fmt_ratio(lift10)}×**, gain top 20% **{c.fmt_pct(gain20)}**. Hai tập có mẫu số khác nhau do bước loại số âm bất thường.",
    "",
    "## Phân khúc và chính sách",
    "",
    "| Phân khúc | Số khách | Tỷ lệ rời bỏ | Doanh thu tháng bị đe dọa | Chính sách đề xuất | KPI |",
    "|---|---:|---:|---:|---|---|",
]
for row in seg.itertuples():
    action, metric = policies[str(row.ActionSegment)]
    report_lines.append(f"| {row.ActionSegment} | {c.fmt_n(row.n)} | {c.fmt_pct(row.rate)} | {c.fmt_money(row.revenue_at_risk)} | {action} | {metric} |")
report_lines += [
    "",
    f"Các nhóm cần can thiệp gồm **{c.fmt_n(intervention.n.sum())} khách**; doanh thu tháng bị đe dọa **{c.fmt_money(intervention.revenue_at_risk.sum())}**.",
    "",
    "Doanh thu tháng bị đe dọa = tổng doanh thu tháng của nhóm × tỷ lệ rời bỏ; chưa phải doanh thu chắc chắn giữ được. Dữ liệu tại một thời điểm không chứng minh nhân quả; chính sách cần thử nghiệm đối chứng.",
]
display(Markdown("\n".join(report_lines)))
